# Run `ts2img-lightcnn` on Google Colab

Notebook đã cập nhật cho repo: `https://github.com/hoangtnedu/ts2img-lightcnn`

Mục tiêu:
- Clone hoặc cập nhật repo từ GitHub.
- Mount Google Drive để lưu checkpoint, backup, log và kết quả.
- Chạy test nhanh trước.
- Chạy thực nghiệm chính.
- Xem file `summary_results.csv`.


## 1. Mount Google Drive

Checkpoint, backup, log và kết quả sẽ được lưu tự động ở:

`/content/drive/MyDrive/ts2img-lightcnn/`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Clone hoặc cập nhật repo từ GitHub

Cell này đã thay `USERNAME` bằng đúng tài khoản GitHub của thầy: `hoangtnedu`.
Nếu thư mục repo đã tồn tại trong `/content`, Colab sẽ `git pull`; nếu chưa có, Colab sẽ `git clone`.


In [ ]:
import os

REPO_URL = "https://github.com/hoangtnedu/ts2img-lightcnn.git"
REPO_DIR = "/content/ts2img-lightcnn"

%cd /content

if os.path.exists(REPO_DIR):
    print("Repo đã tồn tại. Đang cập nhật code mới nhất...")
    %cd $REPO_DIR
    !git pull --rebase
else:
    print("Chưa có repo. Đang clone từ GitHub...")
    !git clone $REPO_URL
    %cd $REPO_DIR

print("Thư mục hiện tại:")
!pwd

print("\nDanh sách file/thư mục trong repo:")
!ls -la


## 3. Cài thư viện

Sau khi cài xong, nếu Colab yêu cầu restart runtime thì chọn **Restart runtime**, sau đó chạy lại từ bước 1.


In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt


## 4. Kiểm tra GPU và môi trường chạy

Nếu dòng `GPU devices` trả về danh sách rỗng `[]`, thầy cần bật GPU tại:

`Runtime > Change runtime type > Hardware accelerator > T4 GPU`


In [ ]:
import os
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Current directory:", os.getcwd())


## 5. Kiểm tra nhanh bằng 2 epoch

Nên chạy cell này trước để kiểm tra toàn bộ pipeline: tải dữ liệu, biến đổi ảnh 2D, tạo mô hình, huấn luyện và lưu kết quả.


In [ ]:
!python -m src.train \
  --dataset GunPoint \
  --representation gaf \
  --model_type light2dcnn \
  --epochs 2 \
  --batch_size 16 \
  --image_size 64 \
  --seed 42


## 6. Chạy một thực nghiệm chính

Sau khi test nhanh chạy ổn, dùng cell này để chạy thực nghiệm chính với `GAF + Light 2D-CNN`.


In [ ]:
!python -m src.train \
  --dataset GunPoint \
  --representation gaf \
  --model_type light2dcnn \
  --epochs 50 \
  --batch_size 32 \
  --image_size 64 \
  --seed 42


## 7. Chạy nhiều thực nghiệm cho bài báo

Cell này sẽ chạy:
- 1D-CNN baseline.
- GAF + Light 2D-CNN.
- MTF + Light 2D-CNN.
- RP + Light 2D-CNN.
- STFT + Light 2D-CNN.
- Lặp lại với các seed: 42, 2024, 2026.

Nếu Colab bị giới hạn GPU, nên giảm `epochs` hoặc chỉ chạy từng nhóm nhỏ.


In [ ]:
!python -m src.run_experiments \
  --datasets GunPoint \
  --representations gaf,mtf,rp,stft \
  --model_type light2dcnn \
  --seeds 42,2024,2026 \
  --epochs 50 \
  --batch_size 32 \
  --image_size 64


## 8. Xem file kết quả

Kết quả tổng hợp nằm ở:

`/content/drive/MyDrive/ts2img-lightcnn/results/summary_results.csv`


In [ ]:
import pandas as pd
from pathlib import Path

result_path = Path("/content/drive/MyDrive/ts2img-lightcnn/results/summary_results.csv")

if result_path.exists():
    df = pd.read_csv(result_path)
    display(df.tail(20))
else:
    print("Chưa thấy file kết quả:", result_path)
    print("Hãy chạy bước 5, 6 hoặc 7 trước.")


## 9. Tải file kết quả về máy

Dùng cell này nếu thầy muốn tải `summary_results.csv` từ Colab về máy tính.


In [ ]:
from google.colab import files
from pathlib import Path

result_path = Path("/content/drive/MyDrive/ts2img-lightcnn/results/summary_results.csv")

if result_path.exists():
    files.download(str(result_path))
else:
    print("Chưa có file để tải:", result_path)
